In [ ]:
import sys, subprocess

def _pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)


try:
    import budoux
except ImportError:
    print("Installing budoux ...")
    _pip_install("budoux")
    import budoux

import json, os, glob, time, math, unicodedata
from collections import defaultdict


try:
    from IPython.display import display, HTML, Markdown
    _RICH = True
except Exception:
    _RICH = False

def show_md(md):
    if _RICH:
        display(Markdown(md))
    else:
        print("\n" + md.replace("`", "").replace("#", "").strip() + "\n")

def show_html(h):
    if _RICH:
        display(HTML(h))
    else:
        print("[rich HTML renders in a notebook; skipped in plain console]")

def rule(title):
    show_md(f"\n---\n## {title}")

CJK_FONT = "'Noto Sans JP','Hiragino Sans','Yu Gothic','Meiryo',sans-serif"
BUDOUX_VERSION = getattr(budoux, "__version__", "unknown")
show_md(f"# BudouX — advanced tutorial\n"
        f"**budoux** `{BUDOUX_VERSION}`  ·  **python** `{sys.version.split()[0]}`")



rule("1 · The problem: CJK & Thai have no spaces, so lines break mid-word")

show_md(
    "English gets free line-break hints from spaces and hyphens. Japanese, Chinese and Thai "
    "do not, so a browser may break a line **inside a word**. BudouX restores meaningful, "
    "resize-safe break points. Here is one sentence in a narrow column with the browser's "
    "naive breaking (notice breaks land anywhere):"
)
demo_ja = "日本語の文章は単語の区切りが明示されないため、狭い画面では単語の途中で改行されてしまいます。"
show_html(f"""
<div style="font-family:{CJK_FONT}; display:flex; gap:24px; align-items:flex-start;">
  <div style="width:150px;border:2px solid #d94040;border-radius:8px;padding:10px;line-height:1.9;">
    <div style="font-size:12px;color:#d94040;font-weight:700;margin-bottom:6px;">✗ naive (breaks anywhere)</div>
    <div style="word-break:normal;overflow-wrap:anywhere;">{demo_ja}</div>
  </div>
</div>
""")



rule("2 · Default parsers for every supported language")

parsers = {
    "Japanese":            ("ja",      budoux.load_default_japanese_parser()),
    "Simplified Chinese":  ("zh-hans", budoux.load_default_simplified_chinese_parser()),
    "Traditional Chinese": ("zh-hant", budoux.load_default_traditional_chinese_parser()),
    "Thai":                ("th",      budoux.load_default_thai_parser()),
}
samples = {
    "Japanese":            "私はコーヒーを飲みながら本を読むのが好きです。",
    "Simplified Chinese":  "我喜欢一边喝咖啡一边读书。",
    "Traditional Chinese": "我喜歡一邊喝咖啡一邊讀書。",
    "Thai":                "ฉันชอบอ่านหนังสือขณะจิบกาแฟ",
}

rows = []
for name, (code, p) in parsers.items():
    phrases = p.parse(samples[name])
    print(f"{name:22s} -> {phrases}")
    rows.append(
        f"<tr><td style='padding:6px 12px;font-weight:600'>{name}</td>"
        f"<td style='padding:6px 12px;color:#999'>{code}</td>"
        f"<td style='padding:6px 12px;text-align:center'>{len(phrases)}</td>"
        f"<td style='padding:6px 12px'>{' │ '.join(phrases)}</td></tr>"
    )
show_html(
    f"<table style='border-collapse:collapse;font-family:{CJK_FONT}'>"
    "<tr style='border-bottom:2px solid #444'>"
    "<th style='text-align:left;padding:6px 12px'>Language</th>"
    "<th style='text-align:left;padding:6px 12px'>Model</th>"
    "<th style='padding:6px 12px'>#phrases</th>"
    "<th style='text-align:left;padding:6px 12px'>Segmentation ( │ = break )</th></tr>"
    + "".join(rows) + "</table>"
)



rule("3 · HTML output & a real line-break demo")

jp = parsers["Japanese"][1]
html_in  = "今日は<b>とても良い天気</b>ですね。"
html_out = jp.translate_html_string(html_in)
show_md(
    "`translate_html_string` preserves your markup and inserts an **invisible** zero-width "
    "space (U+200B) at each phrase boundary, inside a span that only breaks at those points."
)
print("input                    :", html_in)
print("output                   :", html_out)
print("output (ZWSP shown as ·) :", html_out.replace("\u200b", "·"))

long_ja  = "日本語の文章は単語の区切りが明示されないため、狭い画面では単語の途中で改行されてしまいます。"
wrapped  = jp.translate_html_string(long_ja)
show_html(f"""
<div style="font-family:{CJK_FONT}; display:flex; gap:24px; align-items:flex-start;">
  <div style="width:150px;border:2px solid #d94040;border-radius:8px;padding:10px;line-height:1.9;">
    <div style="font-size:12px;color:#d94040;font-weight:700;margin-bottom:6px;">✗ without BudouX</div>
    <div style="word-break:normal;overflow-wrap:anywhere;">{long_ja}</div>
  </div>
  <div style="width:150px;border:2px solid #2aa06a;border-radius:8px;padding:10px;line-height:1.9;">
    <div style="font-size:12px;color:#2aa06a;font-weight:700;margin-bottom:6px;">✓ with BudouX</div>
    {wrapped}
  </div>
</div>
<div style="font-size:12px;color:#999;margin-top:6px">
Both columns are the same width — the green one only breaks between meaningful phrases.</div>
""")

In [ ]:
rule("4 · Look inside the model — it is just a table of weighted features")

models_dir  = os.path.join(os.path.dirname(budoux.__file__), "models")
model_files = sorted(glob.glob(os.path.join(models_dir, "*.json")))
print("Model files shipped with budoux:")
for mf in model_files:
    print(f"  {os.path.basename(mf):12s} {os.path.getsize(mf)/1024:6.1f} KB")

with open(os.path.join(models_dir, "ja.json"), encoding="utf-8") as f:
    ja_model = json.load(f)

groups     = list(ja_model.keys())
n_features = sum(len(v) for v in ja_model.values())
show_md(f"The Japanese model has **{len(groups)} feature groups** and "
        f"**{n_features:,} features** total — yet the whole file is only a few KB.")
for g in groups:
    print(f"  {g:4s}: {len(ja_model[g]):5d} features")

show_md(
    "**Reading a group name.** First letter = n-gram size "
    "(**U**nigram=1 char, **B**igram=2, **T**rigram=3). `W` = the literal character(s). "
    "The digit is the position window around the candidate gap (which sits between 3 and 4):\n\n"
    "```\n"
    "  ... c1  c2  c3 | c4  c5  c6 ...       | = candidate break\n"
    "      UW1 UW2 UW3  UW4 UW5 UW6          single chars\n"
    "        BW1   BW2   BW3                 char pairs\n"
    "      TW1    TW2    TW3    TW4          char triples\n"
    "```\n"
    "A **positive** score pushes toward breaking there; a **negative** score keeps chars together."
)

flat = [((g, feat), score) for g, d in ja_model.items() for feat, score in d.items()]
flat.sort(key=lambda kv: kv[1])
fmt = lambda items: "   ".join(f"{g}:'{feat}'={s:+d}" for (g, feat), s in items)
show_md("**Strongest “break here” features:**")
print(fmt(list(reversed(flat[-8:]))))
show_md("**Strongest “keep together” features:**")
print(fmt(flat[:8]))



rule("5 · Re-implement the segmenter from scratch (and prove it matches)")

show_md(
    "The whole engine is: for each gap, `score = -(sum of all weights) + 2·(sum of matching "
    "feature weights)`; break when `score > 0`. That is it. Below is a faithful reimplementation "
    "— identical windows, guards and arithmetic to the real library."
)

def _boundary_scores(sentence, model):
    """Raw break-score at every gap i (between char i-1 and char i), 1<=i<len."""
    total = sum(sum(group.values()) for group in model.values())
    def g(key, seq):
        return model.get(key, {}).get(seq, 0)
    n, scores = len(sentence), []
    for i in range(1, n):
        s = -total
        if i - 3 >= 0: s += 2 * g("UW1", sentence[i-3:i-2])
        if i - 2 >= 0: s += 2 * g("UW2", sentence[i-2:i-1])
        s += 2 * g("UW3", sentence[i-1:i])
        s += 2 * g("UW4", sentence[i:i+1])
        if i + 1 < n:  s += 2 * g("UW5", sentence[i+1:i+2])
        if i + 2 < n:  s += 2 * g("UW6", sentence[i+2:i+3])
        if i - 2 >= 0: s += 2 * g("BW1", sentence[i-2:i])
        s += 2 * g("BW2", sentence[i-1:i+1])
        if i + 1 < n:  s += 2 * g("BW3", sentence[i:i+2])
        if i - 3 >= 0: s += 2 * g("TW1", sentence[i-3:i])
        if i - 2 >= 0: s += 2 * g("TW2", sentence[i-2:i+1])
        if i + 1 < n:  s += 2 * g("TW3", sentence[i-1:i+2])
        if i + 2 < n:  s += 2 * g("TW4", sentence[i:i+3])
        scores.append(s)
    return scores

def manual_parse(sentence, model):
    """From-scratch equivalent of parser.parse()."""
    if not sentence:
        return []
    result = [sentence[0]]
    for i, s in enumerate(_boundary_scores(sentence, model), start=1):
        if s > 0:
            result.append(sentence[i])
        else:
            result[-1] += sentence[i]
    return result

test_sentences = [
    "私はコーヒーを飲みながら本を読むのが好きです。",
    "機械学習を用いて自然言語処理の問題を解きます。",
    "東京都渋谷区で美味しいラーメンを食べました。",
    "彼女は昨日新しい自転車を買いました。",
    "今日は天気です。",
]
all_ok = True
for s in test_sentences:
    lib, ours = jp.parse(s), manual_parse(s, ja_model)
    ok = lib == ours
    all_ok &= ok
    print(("✓" if ok else "✗"), "│".join(ours))
show_md(f"**From-scratch output matches the library on every test: "
        f"{'✅ PASS' if all_ok else '❌ FAIL'}**")


demo = "私はコーヒーを飲みながら本を読むのが好きです。"
sc   = _boundary_scores(demo, ja_model)
cells = [f"<span style='padding:2px 3px'>{demo[0]}</span>"]
for i, s in enumerate(sc, start=1):
    bar = ("<span style='color:#2aa06a;font-weight:800'>┊</span>" if s > 0
           else "<span style='color:#ddd'>·</span>")
    cells.append(bar)
    cells.append(f"<span style='padding:2px 3px'>{demo[i]}</span>")
show_html(f"<div style='font-family:{CJK_FONT};font-size:24px;line-height:2'>"
          + "".join(cells) + "</div>"
          "<div style='font-size:12px;color:#999'>Green ┊ = model chose to break here.</div>")

In [ ]:
rule("6 · The model is editable — nudge a weight, change the break")

word = "神奈川県"
print("default                :", jp.parse(word))
edited = json.loads(json.dumps(ja_model))


edited.setdefault("UW4", {})["県"] = 10 ** 9
print("after UW4['県'] boosted :", budoux.Parser(edited).parse(word))
show_md(
    "Because the score is a plain linear sum of feature weights, you can hand-tune behaviour, "
    "patch a stubborn word, or **fine-tune** on new data — there is no opaque black box."
)



rule("7 · Train a real, BudouX-compatible model from scratch (AdaBoost)")

show_md(
    "BudouX learns its weights with **AdaBoost** over exactly the UW/BW/TW features above. "
    "Here is a complete (tiny) trainer. It reads phrase-separated text using BudouX's `▁` "
    "(U+2581) convention, extracts the same features, runs AdaBoost, and emits a model dict "
    "you can drop straight into `budoux.Parser`."
)

SEP = "\u2581"
TRAIN = [
    "私は▁コーヒーを▁飲みます", "私は▁本を▁読みます", "彼は▁映画を▁見ます",
    "彼女は▁音楽を▁聴きます", "犬が▁公園で▁走ります", "猫が▁部屋で▁眠ります",
    "母は▁料理を▁作ります", "父は▁新聞を▁読みます", "学生が▁教室で▁勉強します",
    "私は▁毎日▁日本語を▁話します", "彼らは▁東京で▁働きます", "子供が▁お菓子を▁食べます",
]

def to_examples(line):
    """('私は▁本を...') -> (clean_sentence, {indices that start a new phrase})."""
    clean, breaks = [], set()
    for ch in line:
        if ch == SEP:
            if clean:
                breaks.add(len(clean))
        else:
            clean.append(ch)
    return "".join(clean), breaks

def features_at(sentence, i):
    """Exactly the features BudouX consults at the gap before index i (same guards)."""
    n, feats = len(sentence), []
    def add(key, cond, seq):
        if cond:
            feats.append(key + ":" + seq)
    add("UW1", i-3 >= 0, sentence[i-3:i-2]); add("UW2", i-2 >= 0, sentence[i-2:i-1])
    add("UW3", True,     sentence[i-1:i]);   add("UW4", True,     sentence[i:i+1])
    add("UW5", i+1 < n,  sentence[i+1:i+2]); add("UW6", i+2 < n,  sentence[i+2:i+3])
    add("BW1", i-2 >= 0, sentence[i-2:i]);   add("BW2", True,     sentence[i-1:i+1])
    add("BW3", i+1 < n,  sentence[i:i+2])
    add("TW1", i-3 >= 0, sentence[i-3:i]);   add("TW2", i-2 >= 0, sentence[i-2:i+1])
    add("TW3", i+1 < n,  sentence[i-1:i+2]); add("TW4", i+2 < n,  sentence[i:i+3])
    return feats


X, Y = [], []
for line in TRAIN:
    sent, breaks = to_examples(line)
    for i in range(1, len(sent)):
        X.append(set(features_at(sent, i)))
        Y.append(1 if i in breaks else -1)
vocab = sorted({f for feats in X for f in feats})
print(f"{len(X)} boundary examples · {sum(y == 1 for y in Y)} breaks · {len(vocab)} features")

def train_adaboost(X, Y, rounds=200):
    """Minimal AdaBoost; each weak learner = 'is feature f present?' (with polarity)."""
    m = len(Y)
    D = [1.0 / m] * m
    present = {f: set() for f in vocab}
    for k, feats in enumerate(X):
        for f in feats:
            present[f].add(k)
    weight = defaultdict(float)
    for _ in range(rounds):
        posW = sum(D[i] for i in range(m) if Y[i] == 1)
        best_f, best_err, best_pol = None, 1.0, 1
        for f in vocab:
            inS_pos = inS_neg = 0.0
            for i in present[f]:
                if Y[i] == 1: inS_pos += D[i]
                else:         inS_neg += D[i]
            err_pos = inS_neg + (posW - inS_pos)
            err, pol = (err_pos, 1) if err_pos <= 1 - err_pos else (1 - err_pos, -1)
            if err < best_err:
                best_f, best_err, best_pol = f, err, pol
        err   = min(max(best_err, 1e-6), 1 - 1e-6)
        alpha = 0.5 * math.log((1 - err) / err)
        weight[best_f] += alpha * best_pol
        S, Z = present[best_f], 0.0
        for i in range(m):
            pred = best_pol * (1 if i in S else -1)
            D[i] *= math.exp(-alpha * Y[i] * pred)
            Z    += D[i]
        D = [d / Z for d in D]
    return weight

def to_budoux_model(weight, scale=1000):
    """Signed AdaBoost weights -> grouped integer model, ready for budoux.Parser."""
    model = defaultdict(dict)
    for f, w in weight.items():
        group, seq = f.split(":", 1)
        val = int(round(w * scale))
        if val != 0:
            model[group][seq] = val
    return dict(model)

print("Training AdaBoost (toy) ...")
t0 = time.time()
learned   = train_adaboost(X, Y, rounds=200)
toy_model = to_budoux_model(learned)
print(f"done in {time.time()-t0:.2f}s · {sum(len(v) for v in toy_model.values())} weighted features")

toy_parser = budoux.Parser(toy_model)
print("Segmenting with our freshly trained model:")
for s in ["私は本を読みます", "彼女は音楽を聴きます", "父は新聞を読みます", "犬が公園で走ります"]:
    print("  ", "│".join(toy_parser.parse(s)))

encoding_ok = all(
    toy_parser.parse(s) == manual_parse(s, toy_model)
    for s in ["私は本を読みます", "彼女は音楽を聴きます", "学生が教室で勉強します"]
)
show_md(f"Our model loads into the real `budoux.Parser`, and our scorer reproduces it "
        f"byte-for-byte: {'✅' if encoding_ok else '❌'}  "
        f"*(the encoding is genuinely BudouX-compatible)*")

In [ ]:
rule("8 · Practical utilities")

def _char_width(ch):
    return 2 if unicodedata.east_asian_width(ch) in ("W", "F") else 1
def _text_width(s):
    return sum(_char_width(c) for c in s)

def wrap_cjk(text, width, parser):
    """Greedy wrap to `width` display columns, never breaking inside a BudouX phrase."""
    lines, cur, cur_w = [], "", 0
    for phrase in parser.parse(text):
        pw = _text_width(phrase)
        if cur and cur_w + pw > width:
            lines.append(cur); cur, cur_w = "", 0
        cur += phrase; cur_w += pw
    if cur:
        lines.append(cur)
    return lines

para = ("機械学習を用いた自然言語処理は、検索エンジンや翻訳アプリなど、"
        "私たちの日常生活の様々な場面で活躍しています。")
for w in (20, 30):
    show_md(f"**Width-aware wrap to {w} columns** (never splits a phrase):")
    for ln in wrap_cjk(para, w, jp):
        pad = "·" * (w - _text_width(ln))
        print(f"  {ln}{pad}  [{_text_width(ln)}]")

show_md("**Batch → JSON pipeline** (segment many headlines at once):")
headlines = ["台風が接近しています", "新しい駅が来月開業します", "地元のチームが優勝しました"]
pipeline  = [{"text": h, "phrases": jp.parse(h), "html": jp.translate_html_string(h)}
             for h in headlines]
print(json.dumps(pipeline[0], ensure_ascii=False, indent=2))



rule("9 · Micro-benchmark")

t0 = time.time(); _ = budoux.load_default_japanese_parser(); load_ms = (time.time()-t0)*1000
big = [para] * 200
t0 = time.time()
total_chars = 0
for s in big:
    jp.parse(s); total_chars += len(s)
dt = time.time() - t0
print(f"model load  : {load_ms:7.1f} ms")
print(f"parsed      : {len(big)} sentences / {total_chars:,} chars in {dt*1000:7.1f} ms")
print(f"throughput  : {total_chars/max(dt,1e-9):,.0f} chars/sec")

In [2]:
rule("10 · Evaluate segmentation quality (boundary precision / recall / F1)")

GOLD = [
    "私は▁毎朝▁公園を▁散歩します",
    "彼は▁大学で▁物理学を▁研究しています",
    "彼女は▁週末に▁友達と▁買い物に▁行きます",
    "私たちは▁来年▁海外へ▁引っ越す▁予定です",
]
def eval_boundaries(parser, gold_lines):
    tp = fp = fn = 0
    for line in gold_lines:
        sent, gold = to_examples(line)
        pred, pos = set(), 0
        for ph in parser.parse(sent):
            pos += len(ph)
            if pos < len(sent):
                pred.add(pos)
        tp += len(pred & gold); fp += len(pred - gold); fn += len(gold - pred)
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn) if tp + fn else 0.0
    F = 2 * P * R / (P + R) if P + R else 0.0
    return P, R, F, tp, fp, fn

for name, p in [("default ja model", jp), ("our toy model", toy_parser)]:
    P, R, F, tp, fp, fn = eval_boundaries(p, GOLD)
    print(f"{name:18s}  P={P:.2f}  R={R:.2f}  F1={F:.2f}   (tp={tp} fp={fp} fn={fn})")
show_md("The tiny toy model usually scores lower on these unseen sentences — a clean "
        "illustration of why the real models are trained on large corpora (Japanese uses KNBC).")



rule("11 · The command-line interface")

def run(cmd):
    print("$ " + " ".join(cmd))
    try:
        out = subprocess.run(cmd, capture_output=True, text=True)
        print((out.stdout or out.stderr).rstrip())
    except FileNotFoundError:
        print("(the `budoux` command is not on PATH in this environment)")

run(["budoux", "本日は晴天です。"])
run(["budoux", "-l", "zh-hans", "今天天气晴朗。"])
run(["budoux", "-l", "th", "วันนี้อากาศดี"])
run(["budoux", "-H", "本日は晴天です。"])

show_md("---\n**Done.** You have segmented four languages, rendered real HTML breaks, "
        "dissected and edited the model, re-implemented the engine, trained your own "
        "BudouX-compatible model, and benchmarked and evaluated it — all standalone.")

Installing budoux ...


# BudouX — advanced tutorial
**budoux** `0.8.4`  ·  **python** `3.12.13`


---
## 1 · The problem: CJK & Thai have no spaces, so lines break mid-word

English gets free line-break hints from spaces and hyphens. Japanese, Chinese and Thai do not, so a browser may break a line **inside a word**. BudouX restores meaningful, resize-safe break points. Here is one sentence in a narrow column with the browser's naive breaking (notice breaks land anywhere):


---
## 2 · Default parsers for every supported language

...

default ja model    P=1.00  R=0.86  F1=0.92   (tp=12 fp=0 fn=2)
our toy model       P=0.88  R=0.50  F1=0.64   (tp=7 fp=1 fn=7)


The tiny toy model usually scores lower on these unseen sentences — a clean illustration of why the real models are trained on large corpora (Japanese uses KNBC).


---
## 11 · The command-line interface

$ budoux 本日は晴天です。
本日は
晴天です。
$ budoux -l zh-hans 今天天气晴朗。
今天
天气
晴朗。
$ budoux -l th วันนี้อากาศดี
วัน
นี้
อากาศ
ดี
$ budoux -H 本日は晴天です。
<span style="word-break: keep-all; overflow-wrap: anywhere;">本日は​晴天です。</span>


---
**Done.** You have segmented four languages, rendered real HTML breaks, dissected and edited the model, re-implemented the engine, trained your own BudouX-compatible model, and benchmarked and evaluated it — all standalone.